# NanoGPT speedrun — T4 port (`nanogpt-colab-speedrun`)

Single-GPU port of [KellerJordan/modded-nanogpt](https://github.com/KellerJordan/modded-nanogpt) record #360 to a **Colab T4 (16 GB)**.
Target: **≤3.5 canonical-masked FineWeb val cross-entropy** (upstream: 3.28 on 8×H100).

**Before running:** Runtime → Change runtime type → T4 GPU. Then run the cells in order:

1. assert the environment (CUDA torch with Turing kernels)
2. install uv, clone the repo
3. build a `--system-site-packages` venv (reuses Colab's CUDA torch + bundled triton; never installs a second torch)
4. download the data (4×100M train tokens + val, ~800 MB)
5. smoke run (~2 min, small schedule)
6. full run (hours — keep the tab open)

Progress prints every 25 steps, validation every 150, full logs in `logs/<run_id>.txt`.

In [1]:
import torch

assert torch.cuda.is_available(), "No GPU runtime — set Runtime > Change runtime type > T4 GPU"
cap = torch.cuda.get_device_capability()
arch = torch.cuda.get_arch_list()
print(f"GPU: {torch.cuda.get_device_name(0)}  sm{cap[0]}{cap[1]}")
print(f"torch {torch.__version__}  CUDA {torch.version.cuda}  arch: {arch}")
# The port's Triton kernels are tuned for sm75 (T4). The torch build must contain it,
# and the current device must be served by this build (Q9 of the port plan).
assert "sm_75" in arch, f"torch build has no sm_75 (Turing) kernels: {arch}"
dev_arch = f"sm_{cap[0]}{cap[1]}"
assert dev_arch in arch, f"device {dev_arch} not in this build's arch list {arch}"
print("environment OK")

GPU: Tesla T4  sm75
torch 2.11.0+cu128  CUDA 12.8  arch: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
environment OK


In [2]:
import os
import subprocess

subprocess.run(["pip", "install", "-q", "uv"], check=True)
repo = "nanogpt-colab-speedrun"
if not os.path.isdir(repo):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/KingNish24/nanogpt-colab-speedrun.git", repo], check=True)
else:
    # Re-runs must never train stale code: hard-reset the existing clone to the remote tip.
    subprocess.run(["git", "-C", repo, "fetch", "--depth", "1", "origin"], check=True)
    subprocess.run(["git", "-C", repo, "reset", "--hard", "FETCH_HEAD"], check=True)
print("live commit:", subprocess.run(["git", "-C", repo, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())
%cd nanogpt-colab-speedrun

/content/nanogpt-colab-speedrun


In [3]:
# A venv layered on the Colab system environment: torch/triton come from the image,
# the repo's extras (tiktoken, datasets, ...) are installed into the venv.
!uv venv --system-site-packages .venv
!uv pip install --python .venv/bin/python -r requirements.txt
!.venv/bin/python -c "import torch, triton; print('venv OK: torch', torch.__version__, '| triton', triton.__version__, '|', torch.cuda.get_device_name(0))"

Using CPython 3.13.15 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
? A virtual environment already exists at `.venv`. Do you want to replace it? [y/n] › yes

hint: Use the `--clear` flag or set `UV_VENV_CLEAR=1` to skip this prompt^C
Checked 6 packages in 5ms
venv OK: torch 2.11.0+cu128 | triton 3.6.0 | Tesla T4


In [4]:
# 4 x 100M-token train shards + the 1M-token validation shard (~800 MB, from kjj0/fineweb10B-gpt2).
!.venv/bin/python data/cached_fineweb10B.py 4
!du -sh data/fineweb10B && ls -la data/fineweb10B

954M	data/fineweb10B
total 976596
drwxr-xr-x 3 root root      4096 Oct  1 17:57 .
drwxr-xr-x 3 root root      4096 Oct  1 17:57 ..
drwxr-xr-x 3 root root      4096 Oct  1 17:57 .cache
-rw-r--r-- 1 root root 200001024 Oct  1 17:57 fineweb_train_000001.bin
-rw-r--r-- 1 root root 200001024 Oct  1 17:57 fineweb_train_000002.bin
-rw-r--r-- 1 root root 200001024 Oct  1 17:57 fineweb_train_000003.bin
-rw-r--r-- 1 root root 200001024 Oct  1 17:57 fineweb_train_000004.bin
-rw-r--r-- 1 root root 200001024 Oct  1 17:57 fineweb_val_000000.bin


In [6]:
!git -C nanogpt-colab-speedrun pull --ff-only

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 7), reused 8 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 2.24 KiB | 573.00 KiB/s, done.
From https://github.com/KingNish24/nanogpt-colab-speedrun
   408dd5f..ce1b469  master     -> origin/master
Updating 408dd5f..ce1b469
Fast-forward
 port_t4.ipynb                    | 310 ++++++++++++++++++++-------------------
 track_1_short/losses.py          |  23 ++-
 track_1_short/model/gpt.py       |   7 +-
 track_1_short/sampled_softmax.py |   6 +-
 4 files changed, 185 insertions(+), 161 deletions(-)


In [7]:
# Smoke run: the whole pipeline (warmup, stages, sampled softmax, validation, tail averaging,
# final masked eval) on a compressed 8+52+20-step schedule. Expect a few minutes.
import os

os.environ.update(NUM_SCHEDULED_ITERATIONS="8", PYTHONUNBUFFERED="1")
!.venv/bin/python train_gpt.py

logs/6576a8ac-faf7-43c4-b891-f26b3d7dc741.txt
Warming up kernels on steps 0-1 (Triton JIT, SDPA, the fused optimizer), then resetting state
Traceback (most recent call last):
  File "/content/nanogpt-colab-speedrun/train_gpt.py", line 371, in <module>
    main()
    ~~~~^^
  File "/content/nanogpt-colab-speedrun/train_gpt.py", line 262, in main
    train_step(training_manager, model, sampled_softmax, warmup_batches, step, LOSS_SCALE)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/nanogpt-colab-speedrun/train_gpt.py", line 131, in train_step
    sl = sampled_softmax.gather(step, model.lm_head_weight)
  File "/content/nanogpt-colab-speedrun/track_1_short/sampled_softmax.py", line 228, in gather
    self._gather_rows(lm_head_weight, P, inputs)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/nanogpt-colab-speedrun/track_1_short/sampled_softmax.py", line 239, in _gather_rows
    torch.index_select(lm_head_weight,

The smoke run's last lines are the final (canonical-masked) validation — that is the pipeline
working end to end; its `val_loss` is meaningless (8 steps).

The next cell starts the **real run**: 1,080 + 52 growth + 20 extension steps ≈ 287M tokens,
validation every 150 steps, final validation on the tail-averaged weights.
**Keep this tab open** — a disconnected runtime kills the run. Watch for `val_loss` trending
toward ≤3.5; `step_avg` in each log line gives the ETA.

In [ ]:
os.environ.update(NUM_SCHEDULED_ITERATIONS="1080", PYTHONUNBUFFERED="1")
!.venv/bin/python train_gpt.py

In [ ]:
# After the run: the full source-embedded log, and (optionally) keep a copy on Google Drive.
!ls -la logs/
# !cp -r logs /content/drive/MyDrive/nanogpt-colab-speedrun-logs